# Step 1) Feature selection

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, r2_score
from qiskit.circuit.library import pauli_feature_map
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

In [2]:
dataset = pd.read_csv("../dataset/riemann_features.csv")

X = dataset.drop(columns=["distance"])
y = dataset["distance"]

X = X[:1000]
y = y[:1000]

split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Step 2) Training Classical SVR

In [3]:
random_forest_features = pd.read_csv("../results/experiment_11/random_forest_feature_selection.csv")["feature"].tolist()
correlation_features = pd.read_csv("../results/experiment_11/correlation_feature_selection.csv")["feature"].tolist()
gevrey_method_features = pd.read_csv("../results/experiment_11/gevrey_method_feature_selection.csv")["feature"].tolist()
mrmr_10_features = pd.read_csv("../results/experiment_11/mrmr_10_features.csv")["feature"].tolist()
mi_features = pd.read_csv("../results/experiment_11/mi_feature_selection.csv")["feature"].tolist()

features_map = {
    "Random Forest": random_forest_features[:10],
    "Correlation": correlation_features[:10],
    "Gevrey Method": gevrey_method_features[:10],
    "MRMR (10 features)": mrmr_10_features,
    "Mutual Information": mi_features,
    "My Selection":[
        "z_co_gram_lag_2",
        "z_gram",
        "z_gram_lag_1",
        "d_lag_13",
        "z_co_gram_lag_3",
        "d_lag_12",
        "d_lag_14",
        "d_lag_1",
        "z_co_gram_lag_1",
        "z_gram_lag_2"
    ]
}

In [4]:
def run_classical_experiment(features_map):
    results = []

    param_grid = {
        "svr__C": [0.1, 1, 10, 100],
        "svr__epsilon": [0.001, 0.01, 0.1, 0.5],
        "svr__gamma": ["scale", 0.01, 0.1, 1.0]
    }

    for group_name, features in features_map.items():
        print(f"Running experiment for group: {group_name} with {len(features)} features")

        X_train_subset = X_train[features].to_numpy()
        X_test_subset = X_test[features].to_numpy()

        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("svr", SVR(kernel="rbf"))
        ])

        grid = GridSearchCV(
            pipeline,
            param_grid,
            scoring="neg_root_mean_squared_error",
            cv=TimeSeriesSplit(n_splits=5),
            n_jobs=-1
        )

        grid.fit(X_train_subset, y_train)

        best_model = grid.best_estimator_

        y_pred = best_model.predict(X_test_subset)

        rmse = root_mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        results.append({
            "Group": group_name,
            "RMSE": rmse,
            "R2": r2,
            "Best C": grid.best_params_["svr__C"],
            "Best epsilon": grid.best_params_["svr__epsilon"],
            "Best gamma": grid.best_params_["svr__gamma"],
            "Features": len(features)
        })

    return pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)


In [5]:
df_standard = run_classical_experiment(features_map)
print(df_standard)
df_standard.to_csv("../results/experiment_11/classical_results.csv", index=False)

Running experiment for group: Random Forest with 10 features


Running experiment for group: Correlation with 10 features
Running experiment for group: Gevrey Method with 10 features
Running experiment for group: MRMR (10 features) with 10 features
Running experiment for group: Mutual Information with 10 features
Running experiment for group: My Selection with 10 features
                Group      RMSE        R2  Best C  Best epsilon  Best gamma  \
0        My Selection  0.072092  0.941185      10         0.001        0.10   
1  Mutual Information  0.099188  0.888666     100         0.010        0.01   
2  MRMR (10 features)  0.108122  0.867707     100         0.010        0.01   
3       Random Forest  0.188140  0.599437     100         0.100        0.01   
4       Gevrey Method  0.191510  0.584958      10         0.010        0.01   
5         Correlation  0.262881  0.217962       1         0.100        0.01   

   Features  
0        10  
1        10  
2        10  
3        10  
4        10  
5        10  


# Step 3) Traning Quantum SVR using the same best hyperparameters and features of classical SVR using Angle Embedding

In [6]:
dev = qml.device("default.qubit", wires=10)

@qml.qnode(dev)
def kernel(x1, x2, n_qubits):
    qml.AngleEmbedding(x1, wires=range(n_qubits), rotation="X")
    qml.adjoint(qml.AngleEmbedding)(x2, wires=range(n_qubits), rotation="X")
    return qml.expval(qml.Projector([0] * n_qubits, wires=range(n_qubits)))


def kernel_mat(A, B):
    mat = []
    for a in A:
        row = []
        for b in B:
            row.append(kernel(a, b, n_qubits=10))
        mat.append(row)
    return np.array(mat)


results = []
for group_name, features in features_map.items():
    print(f"Running quantum experiment for group: {group_name} with {len(features)} features")
    scaler = StandardScaler()
    X_train_subset = X_train[features].to_numpy()
    X_train_scaled = scaler.fit_transform(X_train_subset)
    X_test_subset = X_test[features].to_numpy()
    X_test_scaled = scaler.transform(X_test_subset)

    svr = SVR(kernel=kernel_mat, C=10)
    svr.fit(X_train_scaled, y_train)
    y_pred = svr.predict(X_test_scaled)

    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append( {
        "Group": f"Quantum Kernel SVR - {group_name} Features",
        "RMSE": rmse,
        "R2": r2,
        "qubits": 10,
    })

df_quantum = pd.DataFrame(results)
print(df_quantum)
df_quantum.to_csv("../results/experiment_11/quantum_angle_embedding_results.csv", index=False)

Running quantum experiment for group: Random Forest with 10 features
Running quantum experiment for group: Correlation with 10 features
Running quantum experiment for group: Gevrey Method with 10 features
Running quantum experiment for group: MRMR (10 features) with 10 features
Running quantum experiment for group: Mutual Information with 10 features
Running quantum experiment for group: My Selection with 10 features
                                              Group      RMSE        R2  \
0       Quantum Kernel SVR - Random Forest Features  0.234103  0.379815   
1         Quantum Kernel SVR - Correlation Features  0.291322  0.039592   
2       Quantum Kernel SVR - Gevrey Method Features  0.327355 -0.212682   
3  Quantum Kernel SVR - MRMR (10 features) Features  0.157111  0.720669   
4  Quantum Kernel SVR - Mutual Information Features  0.138222  0.783795   
5        Quantum Kernel SVR - My Selection Features  0.120339  0.836120   

   qubits  
0      10  
1      10  
2      10  
3    

In [7]:
dev = qml.device("default.qubit", wires=10)

@qml.qnode(dev)
def kernel(x1, x2, n_qubits):
    qml.AngleEmbedding(x1, wires=range(n_qubits), rotation="Y")
    qml.adjoint(qml.AngleEmbedding)(x2, wires=range(n_qubits), rotation="Y")
    return qml.expval(qml.Projector([0] * n_qubits, wires=range(n_qubits)))


def kernel_mat(A, B):
    mat = []
    for a in A:
        row = []
        for b in B:
            row.append(kernel(a, b, n_qubits=10))
        mat.append(row)
    return np.array(mat)


results = []
for group_name, features in features_map.items():
    print(f"Running quantum experiment for group: {group_name} with {len(features)} features")
    scaler = StandardScaler()
    X_train_subset = X_train[features].to_numpy()
    X_train_scaled = scaler.fit_transform(X_train_subset)
    X_test_subset = X_test[features].to_numpy()
    X_test_scaled = scaler.transform(X_test_subset)

    svr = SVR(kernel=kernel_mat, C=10)
    svr.fit(X_train_scaled, y_train)
    y_pred = svr.predict(X_test_scaled)

    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append( {
        "Group": f"Quantum Kernel SVR - {group_name} Features",
        "RMSE": rmse,
        "R2": r2,
        "qubits": 10,
    })

df_quantum = pd.DataFrame(results)
print(df_quantum)
df_quantum.to_csv("../results/experiment_11/quantum_angle_embedding_results2.csv", index=False)

Running quantum experiment for group: Random Forest with 10 features
Running quantum experiment for group: Correlation with 10 features
Running quantum experiment for group: Gevrey Method with 10 features
Running quantum experiment for group: MRMR (10 features) with 10 features
Running quantum experiment for group: Mutual Information with 10 features
Running quantum experiment for group: My Selection with 10 features
                                              Group      RMSE        R2  \
0       Quantum Kernel SVR - Random Forest Features  0.234103  0.379815   
1         Quantum Kernel SVR - Correlation Features  0.291322  0.039592   
2       Quantum Kernel SVR - Gevrey Method Features  0.327355 -0.212682   
3  Quantum Kernel SVR - MRMR (10 features) Features  0.157111  0.720669   
4  Quantum Kernel SVR - Mutual Information Features  0.138222  0.783795   
5        Quantum Kernel SVR - My Selection Features  0.120339  0.836120   

   qubits  
0      10  
1      10  
2      10  
3    

In [ ]:
dev = qml.device("default.qubit", wires=10)

@qml.qnode(dev)
def kernel(x1, x2, n_qubits):
    qml.AngleEmbedding(x1, wires=range(n_qubits), rotation="Z")
    qml.adjoint(qml.AngleEmbedding)(x2, wires=range(n_qubits), rotation="Z")
    return qml.expval(qml.Projector([0] * n_qubits, wires=range(n_qubits)))


def kernel_mat(A, B):
    mat = []
    for a in A:
        row = []
        for b in B:
            row.append(kernel(a, b, n_qubits=10))
        mat.append(row)
    return np.array(mat)


results = []
for group_name, features in features_map.items():
    print(f"Running quantum experiment for group: {group_name} with {len(features)} features")
    scaler = StandardScaler()
    X_train_subset = X_train[features].to_numpy()
    X_train_scaled = scaler.fit_transform(X_train_subset)
    X_test_subset = X_test[features].to_numpy()
    X_test_scaled = scaler.transform(X_test_subset)

    svr = SVR(kernel=kernel_mat, C=10)
    svr.fit(X_train_scaled, y_train)
    y_pred = svr.predict(X_test_scaled)

    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append( {
        "Group": f"Quantum Kernel SVR - {group_name} Features",
        "RMSE": rmse,
        "R2": r2,
        "qubits": 10,
    })

df_quantum = pd.DataFrame(results)
print(df_quantum)
df_quantum.to_csv("../results/experiment_11/quantum_angle_embedding_results3.csv", index=False)

Running quantum experiment for group: Random Forest with 10 features
Running quantum experiment for group: Correlation with 10 features
Running quantum experiment for group: Gevrey Method with 10 features
Running quantum experiment for group: MRMR (10 features) with 10 features
Running quantum experiment for group: Mutual Information with 10 features
Running quantum experiment for group: My Selection with 10 features
                                              Group      RMSE        R2  \
0       Quantum Kernel SVR - Random Forest Features  0.298474 -0.008142   
1         Quantum Kernel SVR - Correlation Features  0.298474 -0.008142   
2       Quantum Kernel SVR - Gevrey Method Features  0.298474 -0.008142   
3  Quantum Kernel SVR - MRMR (10 features) Features  0.298474 -0.008142   
4  Quantum Kernel SVR - Mutual Information Features  0.298474 -0.008142   
5        Quantum Kernel SVR - My Selection Features  0.298474 -0.008142   

   qubits  
0      10  
1      10  
2      10  
3    

# Step 4) Traning Quantum SVR using the same best hyperparameters and features of classical SVR using PauliFeatureMap

In [4]:
def create_quantum_kernel(n_qubits, reps=3, entanglement="full", paulis=["ZZ", "Z"]):
    feature_map = pauli_feature_map(feature_dimension=n_qubits, reps=reps, entanglement=entanglement, paulis=paulis)
    sampler = StatevectorSampler()
    fidelity = ComputeUncompute(sampler=sampler)
    quantum_kernel = FidelityQuantumKernel(feature_map=feature_map, fidelity=fidelity)
    return quantum_kernel

In [ ]:
results = []

print("Running quantum experiments with Pauli Feature Map\n")

for group_name, features in features_map.items():
    if group_name != "My Selection":
        continue
    print(f"Running quantum experiment with Pauli Feature Map for group: {group_name} with {len(features)} features")
    scaler = StandardScaler()
    X_train_subset = X_train[features].to_numpy()
    X_train_scaled = scaler.fit_transform(X_train_subset)
    X_test_subset = X_test[features].to_numpy()
    X_test_scaled = scaler.transform(X_test_subset)

    quantum_kernel = create_quantum_kernel(n_qubits=10, paulis=["X", "Y"], entanglement='linear')
    svr = SVR(kernel=quantum_kernel.evaluate, C=10, epsilon=0.01)
    svr.fit(X_train_scaled, y_train)
    y_pred = svr.predict(X_test_scaled)

    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Group": f"Quantum Kernel SVR - {group_name} Features",
        "RMSE": rmse,
        "R2": r2,
        "qubits": 10,
    })


df_quantum = pd.DataFrame(results)
print(df_quantum)
df_quantum.to_csv("../results/experiment_11/quantum_pauli_feature_map_results.csv", index=False)

Running quantum experiments with Pauli Feature Map

Running quantum experiment with Pauli Feature Map for group: My Selection with 10 features
                                        Group      RMSE        R2  qubits
0  Quantum Kernel SVR - My Selection Features  0.298586 -0.008899      10


In [6]:
results = []

print("Running quantum experiments with Pauli Feature Map\n")

for group_name, features in features_map.items():
    if group_name != "My Selection":
        continue
    print(f"Running quantum experiment with Pauli Feature Map for group: {group_name} with {len(features)} features")
    scaler = StandardScaler()
    X_train_subset = X_train[features].to_numpy()
    X_train_scaled = scaler.fit_transform(X_train_subset)
    X_test_subset = X_test[features].to_numpy()
    X_test_scaled = scaler.transform(X_test_subset)

    quantum_kernel = create_quantum_kernel(n_qubits=10, paulis=["X", "Y", "Z", "ZZ"], entanglement='linear', reps=2)
    svr = SVR(kernel=quantum_kernel.evaluate, C=10, epsilon=0.01)
    svr.fit(X_train_scaled, y_train)
    y_pred = svr.predict(X_test_scaled)

    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Group": f"Quantum Kernel SVR - {group_name} Features",
        "RMSE": rmse,
        "R2": r2,
        "qubits": 10,
    })

df_quantum = pd.DataFrame(results)
print(df_quantum)
df_quantum.to_csv("../results/experiment_11/quantum_pauli_feature_map_results_2.csv", index=False)

Running quantum experiments with Pauli Feature Map

Running quantum experiment with Pauli Feature Map for group: My Selection with 10 features
                                        Group      RMSE        R2  qubits
0  Quantum Kernel SVR - My Selection Features  0.294066  0.021415      10
